## Edge bundling failed experiment

In [ ]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import h3
from shapely.geometry import Polygon, Point
import matplotlib.pyplot as plt
import mapclassify
from shapely import Polygon
import shapely
import contextily
from shapely.geometry import box
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import plotly.express as px
import seaborn as sns

In [ ]:
mobility_europe = []

In [ ]:
mobility_europe["start"] = mobility_europe.geometry.apply(lambda g: Point(g.coords[0]))
mobility_europe["end"]   = mobility_europe.geometry.apply(lambda g: Point(g.coords[-1]))
mobility_europe = mobility_europe.to_crs(4326)
mobility_europe.head(2)

In [ ]:
import math

class Edge:

    def __init__(self, source, destination):
        self.source = source
        self.destination = destination
        self.distance = -1
        self.weight = -1
        self.skip = False
        self.lock = False


class Node:

    def __init__(self, id, longitude, latitude, name):
        self.id = id
        self.longitude = longitude
        self.latitude = latitude
        self.name = name
        self.edges = []

        # dijkstra related attributes
        self.distance = -1
        self.visited = False
        self.previous = None
        self.previous_edge = None

    def distance_to(self, other):
        return math.sqrt(pow(other.longitude - self.longitude, 2) + pow(other.latitude - self.latitude, 2))

    def __lt__(self, other):
        return self.id < other.id

    def __gt__(self, other):
        return self.id > other.id

    def __le__(self, other):
        return self.id <= other.id

    def __ge__(self, other):
        return self.id >= other.id

In [ ]:
nodes = {}
edges = []

# extract unique node geometries (start + end)
start_nodes = mobility_europe[["ORIGIN", "start"]].rename(columns={"start": "pt"})
end_nodes = mobility_europe[["DESTINATION", "end"]].rename(columns={"DESTINATION": "ORIGIN", "end": "pt"})
all_nodes = pd.concat([start_nodes, end_nodes]).drop_duplicates("ORIGIN")

# create Node instances
for _, row in all_nodes.iterrows():
    node_id = row["ORIGIN"]
    pt = row["pt"]

    nodes[node_id] = Node(
        id=node_id,
        longitude=pt.x,
        latitude=pt.y,
        name=node_id
    )

edge_counts = mobility_europe.groupby(["ORIGIN", "DESTINATION"]).size().reset_index(name="weight")

for _, row in edge_counts.iterrows():
    o = row["ORIGIN"]
    d = row["DESTINATION"]

    e = Edge(o, d)
    e.distance = nodes[o].distance_to(nodes[d])
    e.weight = row["weight"]

    edges.append(e)
    nodes[o].edges.append(e)

In [ ]:
import heapq

# Dijsktra
def find_shortest_path(source, dest, nodes):
    # reset nodes
    for node in nodes.values():
        node.distance = 9999999999999999999
        node.visited = False
        node.previous = None
        node.previous_edge = None

    source.distance = 0
    queue = []
    heapq.heappush(queue, (source.distance, source))

    while not len(queue) == 0:
        next_node = heapq.heappop(queue)[1]
        next_node.visited = True

        for edge in next_node.edges:
            if edge.skip:
                continue

            other_id = edge.destination if edge.source == next_node.id else edge.source
            other = nodes[other_id]

            current_distance = next_node.distance + edge.weight

            if current_distance < other.distance:
                other.distance = current_distance
                other.previous = next_node
                other.previous_edge = edge
                heapq.heappush(queue, (other.distance, other))

        # Already found the destination, no need to continue with the search
        if next_node == dest:
            break

    # extract path
    path = []

    node = dest
    while node.previous is not None:
        path.append(node.previous_edge)
        node = node.previous

    path.reverse()
    return path

In [ ]:
# Bezier

def eval_bezier(control_points, t):

    if len(control_points) < 2:
        return np.zeros(2)

    if t < 0 or t > 1:
        return np.zeros(2)
    if t == 0:
        return control_points[0]
    if t == 1:
        return control_points[-1]

    # Calculate the intermediate points
    points = control_points
    while len(points) > 1:
        intermediate_points = []
        for i in range(len(points)-1):
            p = (1 - t) * points[i] + t * points[i+1]
            intermediate_points.append(p)
        points = intermediate_points
    return points[0]

def create_bezier_polygon(control_points, n):

    if n < 2:
        return [control_points[0], control_points[-1]]
    points = []
    for i in range(n):
        points.append(eval_bezier(control_points, i/n))
    points.append(control_points[-1])
    return points

def get_control_points(source, dest, nodes, path, smoothing):
    control_points = []
    current_node = source
    for edge_in_path in path:
        control_points.append(np.array([current_node.longitude, current_node.latitude]))

        other_node_id = edge_in_path.destination if edge_in_path.source == current_node.id else edge_in_path.source
        current_node = nodes[other_node_id]

    control_points.append(np.array([dest.longitude, dest.latitude]))
    return split(control_points, smoothing)

def split(points, smoothing):
    p = points
    # for each level of smoothing, insert new control point in the middle of all points already in the array
    # loop starts from 1 => smoothing level of 1 keeps only node points
    for smooth in range(1, smoothing):
        new_points = []
        for i in range(len(p) - 1):
            new_point = np.divide(p[i] + p[i + 1], 2.0)
            new_points.append(p[i])
            new_points.append(new_point)
        new_points.append(p[-1])
        p = new_points
    return p

In [ ]:
d = 8.0  # edge weight parameter, weight = distance^d
k = 3.0  # if new detour is longer than k times the original line it is not bundled
n = 100  # number of sampling points in Bezier curves
smoothing = 2  # bezier smoothing

In [ ]:
control_point_lists = []
too_long = 0
no_path = 0
for edge in edges:
    if edge.lock:
        continue

    edge.skip = True

    source = nodes[edge.source]
    dest = nodes[edge.destination]

    edge.weight = pow(edge.distance, d)
    # print(edge.distance, edge.weight)
    path = find_shortest_path(source, dest, nodes)

    if len(path) == 0:
        no_path += 1
        edge.skip = False
        continue

    original_edge_distance = source.distance_to(dest)
    new_path_length = sum([e.distance for e in path])

    if new_path_length > k * original_edge_distance:
        too_long += 1
        edge.skip = False
        continue

    for edge_in_path in path:
        edge_in_path.lock = True

    control_point_lists.append(get_control_points(source, dest, nodes, path, smoothing))


In [ ]:
from pyproj import Transformer
transformer = Transformer.from_crs(4326, 3035, always_xy=True)
fig, ax = plt.subplots(figsize=(12, 13), facecolor='#1a1a2e')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

bezier_polygons = []
for controlPoints in control_point_lists:
    polygon = create_bezier_polygon(controlPoints, n) 
    bezier_polygons.append(polygon)
    points = [transformer.transform(arr[0], arr[1]) for arr in polygon]
    x, y = zip(*points)
    plt.plot(x, y, color='indigo', linewidth=0.4, alpha=0.2)
    
contextily.add_basemap(
    ax,
    source=contextily.providers.OpenStreetMap.Mapnik,
    crs=3035,
    zoom=5,
    alpha=1
)

# plt.savefig(ASSETS_DIR / "edgebundled_map.png", dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()